In [2]:
import nflreadpy as nfl
# Load weekly player stats for 2019-2025
player_stats = nfl.load_player_stats(range(2019,2026))

# Convert from Polars to pandas
df = player_stats.to_pandas()

# Only use regular-season games for fantasy modeling
df = df[df['season_type'] == 'REG'].copy()

print(df.shape)
print(df['position'].value_counts())
print(df['season'].unique())

(124022, 150)
position
WR     16859
LB     14756
CB     12708
RB     10604
DE     10368
DT      9668
TE      8377
SAF     6015
QB      4490
DB      4320
K       3737
P       3691
OT      3020
OLB     2936
FS      2389
G       2051
S       1746
ILB     1468
MLB     1356
C       1040
NT       872
FB       715
LS       465
DL       218
OL        29
Name: count, dtype: int64
[2019 2020 2021 2022 2023 2024 2025]


In [3]:
# ----------- WR/TE Feature Engineering -----------

# Past 3 game averages
df['targets_avg_3'] = df.groupby(['player_id', 'season'])['targets'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_avg_3'] = df.groupby(['player_id', 'season'])['receptions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_yards_avg_3'] = df.groupby(['player_id', 'season'])['receiving_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['target_share_avg_3'] = df.groupby(['player_id','season'])['target_share'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_air_yards_avg_3'] = df.groupby(['player_id','season'])['receiving_air_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['air_yards_share_avg_3'] = df.groupby(['player_id','season'])['air_yards_share'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_td_avg_3'] = df.groupby(['player_id','season'])['receiving_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rec_yards_after_catch_avg_3'] = df.groupby(['player_id','season'])['receiving_yards_after_catch'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['targets_avg_5'] = df.groupby(['player_id', 'season'])['targets'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_avg_5'] = df.groupby(['player_id', 'season'])['receptions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_yards_avg_5'] = df.groupby(['player_id', 'season'])['receiving_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['target_share_avg_5'] = df.groupby(['player_id','season'])['target_share'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_air_yards_avg_5'] = df.groupby(['player_id','season'])['receiving_air_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['air_yards_share_avg_5'] = df.groupby(['player_id','season'])['air_yards_share'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_td_avg_5'] = df.groupby(['player_id','season'])['receiving_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rec_yards_after_catch_avg_5'] = df.groupby(['player_id','season'])['receiving_yards_after_catch'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['targets_trend'] = df['targets_avg_3'] - df['targets_avg_5']
df['target_share_trend'] = df['target_share_avg_3'] - df['target_share_avg_5']
df['rec_yards_trend'] = df['rec_yards_avg_3'] - df['rec_yards_avg_5']
df['rec_air_yards_trend'] = df['rec_air_yards_avg_3'] - df['rec_air_yards_avg_5']



In [4]:
# ----------- RB Feature Engineering -----------

df['opportunities'] = df['carries'] + df['targets']

# Past 3 game averages
df['carries_avg_3'] = df.groupby(['player_id','season'])['carries'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rushing_yards_avg_3'] = df.groupby(['player_id','season'])['rushing_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['rushing_tds_avg_3'] = df.groupby(['player_id','season'])['rushing_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['opportunities_avg_3'] = df.groupby(['player_id','season'])['opportunities'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())


# Past 5 game averages
df['carries_avg_5'] = df.groupby(['player_id','season'])['carries'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rushing_yards_avg_5'] = df.groupby(['player_id','season'])['rushing_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['rushing_tds_avg_5'] = df.groupby(['player_id','season'])['rushing_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['opportunities_avg_5'] = df.groupby(['player_id','season'])['opportunities'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['carries_trend'] = df['carries_avg_3'] - df['carries_avg_5']
df['rushing_yards_trend'] = df['rushing_yards_avg_3'] - df['rushing_yards_avg_5']
df['opportunities_trend'] = df['opportunities_avg_3'] - df['opportunities_avg_5']


In [5]:
# ----------- QB Feature Engineering -----------

# Past 3 game averages
df['completions_avg_3'] = df.groupby(['player_id', 'season'])['completions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['attempts_avg_3'] = df.groupby(['player_id', 'season'])['attempts'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_yards_avg_3'] = df.groupby(['player_id', 'season'])['passing_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_tds_avg_3'] = df.groupby(['player_id', 'season'])['passing_tds'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_int_avg_3'] = df.groupby(['player_id', 'season'])['passing_interceptions'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_air_yards_avg_3'] = df.groupby(['player_id', 'season'])['passing_air_yards'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['passing_first_downs_avg_3'] = df.groupby(['player_id', 'season'])['passing_first_downs'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['completions_avg_5'] = df.groupby(['player_id', 'season'])['completions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['attempts_avg_5'] = df.groupby(['player_id', 'season'])['attempts'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_yards_avg_5'] = df.groupby(['player_id', 'season'])['passing_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_tds_avg_5'] = df.groupby(['player_id', 'season'])['passing_tds'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_int_avg_5'] = df.groupby(['player_id', 'season'])['passing_interceptions'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_air_yards_avg_5'] = df.groupby(['player_id', 'season'])['passing_air_yards'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['passing_first_downs_avg_5'] = df.groupby(['player_id', 'season'])['passing_first_downs'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['attempts_trend'] = df['attempts_avg_3'] - df['attempts_avg_5']
df['passing_yards_trend'] = df['passing_yards_avg_3'] - df['passing_yards_avg_5']
df['passing_air_yards_trend'] = df['passing_air_yards_avg_3'] - df['passing_air_yards_avg_5']


In [6]:
# ----------- K Feature Engineering -----------

# Past 3 game averages
df['fg_att_avg_3'] = df.groupby(['player_id', 'season'])['fg_att'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_made_avg_3'] = df.groupby(['player_id', 'season'])['fg_made'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_long_avg_3'] = df.groupby(['player_id', 'season'])['fg_long'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['fg_made_50_59_avg_3'] = df.groupby(['player_id', 'season'])['fg_made_50_59'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['pat_att_avg_3'] = df.groupby(['player_id', 'season'])['pat_att'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())
df['pat_made_avg_3'] = df.groupby(['player_id', 'season'])['pat_made'].transform(lambda x : x.shift(1).rolling(window=3, min_periods=1).mean())

# Past 5 game averages
df['fg_att_avg_5'] = df.groupby(['player_id', 'season'])['fg_att'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_made_avg_5'] = df.groupby(['player_id', 'season'])['fg_made'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_long_avg_5'] = df.groupby(['player_id', 'season'])['fg_long'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['fg_made_50_59_avg_5'] = df.groupby(['player_id', 'season'])['fg_made_50_59'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['pat_att_avg_5'] = df.groupby(['player_id', 'season'])['pat_att'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())
df['pat_made_avg_5'] = df.groupby(['player_id', 'season'])['pat_made'].transform(lambda x : x.shift(1).rolling(window=5, min_periods=1).mean())

# Trends
df['fg_att_trend'] = df['fg_att_avg_3'] - df['fg_att_avg_5']
df['fg_made_trend'] = df['fg_made_avg_3'] - df['fg_made_avg_5']

# Kicker fantasy points
df['kicker_fantasy_points'] = (
    3 * df['fg_made'] +
    1 * df['pat_made']
)


In [10]:
# ----------- Opponent Matchup Feature Engineering -----------

fantasy_positions = ['QB', 'RB', 'WR', 'TE', 'K']

matchup_df = df[
    df['position'].isin(fantasy_positions)
].copy()

matchup_df['matchup_points'] = matchup_df['fantasy_points_ppr']

matchup_df.loc[
    matchup_df['position'] == 'K',
    'matchup_points'
] = matchup_df.loc[
    matchup_df['position'] == 'K',
    'kicker_fantasy_points'
]

weekly_points_allowed = (
    matchup_df.groupby(
        ['season', 'week', 'opponent_team', 'position'],
        as_index=False
    )['matchup_points']
    .sum()
)

weekly_points_allowed = weekly_points_allowed.rename(
    columns = {
        'matchup_points': 'points_allowed'
    }
)

weekly_points_allowed = weekly_points_allowed.sort_values(
    ['opponent_team', 'position', 'season', 'week']
).reset_index(drop=True)

matchup_group = [
    'opponent_team',
    'position',
    'season'
]

weekly_points_allowed['opp_points_allowed_avg_3'] = (
    weekly_points_allowed
    .groupby(matchup_group)['points_allowed']
    .transform(
        lambda x:
        x.shift(1)
        .rolling(window=3, min_periods=1)
        .mean()
    )
)

weekly_points_allowed['opp_points_allowed_avg_5'] = (
    weekly_points_allowed
    .groupby(matchup_group)['points_allowed']
    .transform(
        lambda x:
        x.shift(1)
        .rolling(window=5, min_periods=1)
        .mean()
    )
)

weekly_points_allowed['opp_points_allowed_trend'] = (
    weekly_points_allowed['opp_points_allowed_avg_3']
    - weekly_points_allowed['opp_points_allowed_avg_5']
)

matchup_features = weekly_points_allowed[
    [
        'season',
        'week',
        'opponent_team',
        'position',
        'opp_points_allowed_avg_3',
        'opp_points_allowed_avg_5',
        'opp_points_allowed_trend'
    ]
]

df = df.merge(
    matchup_features,
    on=[
        'season',
        'week',
        'opponent_team',
        'position'
    ],
    how='left'
)

print(
    df[
        (df['position'] == 'WR') &
        (df['week'] > 1)
    ][
        [
            'player_display_name',
            'season',
            'week',
            'opponent_team',
            'opp_points_allowed_avg_3',
            'opp_points_allowed_avg_5',
            'opp_points_allowed_trend'
        ]
    ].head(15)
)

KeyError: 'matchup_points'

In [7]:
print(
    df[
        (df['position'] == 'WR') &
        (df['week'] > 1)
    ][
        ['player_display_name', 'season', 'week',
         'targets_avg_3', 'targets_avg_5', 'targets_trend']
    ].head(10)
)

     player_display_name  season  week  targets_avg_3  targets_avg_5  \
1038    Larry Fitzgerald    2019     2           13.0           13.0   
1070      Danny Amendola    2019     2           13.0           13.0   
1077      Matthew Slater    2019     2            NaN            NaN   
1085    Michael Crabtree    2019     2            NaN            NaN   
1095      Julian Edelman    2019     2           11.0           11.0   
1102    Emmanuel Sanders    2019     2            7.0            7.0   
1108       Antonio Brown    2019     2            NaN            NaN   
1116    Demaryius Thomas    2019     2            NaN            NaN   
1124         Julio Jones    2019     2           11.0           11.0   
1135        Randall Cobb    2019     2            5.0            5.0   

      targets_trend  
1038            0.0  
1070            0.0  
1077            NaN  
1085            NaN  
1095            0.0  
1102            0.0  
1108            NaN  
1116            NaN  
1124     

In [8]:
# To Parquet
df.to_parquet(
    "../data/processed/player_features.parquet",
    index=False
)